In [1]:
import h5py
import pandas as pd

archivo_path = 'Data/datos_nsrdb/nsrdb_puerto_rico_1998.h5'

with h5py.File(archivo_path, 'r') as f:
    # 1. Listar los datasets disponibles dentro del archivo
    print("Datasets disponibles en el archivo HDF5:")
    print(list(f.keys()))
    
    # 2. Inspeccionar la dimensión de una variable (ej. GHI)
    # Estructura típica: (Pasos de tiempo, Número de ubicaciones/píxeles)
    shape_ghi = f['ghi'].shape
    print(f"\nDimensión del dataset 'ghi': {shape_ghi}")
    
    # 3. Leer los metadatos espaciales (Latitud, Longitud, Elevación)
    # El dataset 'meta' suele contener la información de los nodos/coordenadas
    meta_df = pd.DataFrame(f['meta'][:])
    print("\nEstructura de la tabla de coordenadas ('meta'):")
    print(meta_df.head())
    
    # 4. Leer el índice de tiempo
    time_index = f['time_index'][:]
    print(f"\nTotal de registros temporales: {len(time_index)}")
    print(f"Primeros registros: {time_index[:3]}")

Datasets disponibles en el archivo HDF5:
['air_temperature', 'clearsky_dhi', 'clearsky_dni', 'clearsky_ghi', 'coordinates', 'dhi', 'dni', 'ghi', 'meta', 'solar_zenith_angle', 'surface_albedo', 'surface_pressure', 'time_index', 'total_precipitable_water', 'wind_speed']

Dimensión del dataset 'ghi': (105120, 2480)

Estructura de la tabla de coordenadas ('meta'):
    latitude  longitude  elevation
0  18.120001 -67.930000       56.0
1  18.100000 -67.930000       63.0
2  18.080000 -67.930000       37.0
3  18.059999 -67.930000        8.0
4  18.120001 -67.910004       71.0

Total de registros temporales: 105120
Primeros registros: [b'1998-01-01 00:00:00+00:00' b'1998-01-01 00:05:00+00:00'
 b'1998-01-01 00:10:00+00:00']


In [3]:
import h5py
import numpy as np
import pandas as pd

def extraer_tensor_espaciotemporal(h5_path, variables, start_idx=0, end_idx=288):
    """
    Extrae variables del HDF5, aplica el factor de escala y devuelve un DataFrame
    estructurado por [Tiempo, Nodo].
    """
    datos_extraidos = {}
    
    with h5py.File(h5_path, 'r') as f:
        # Extraer índice de tiempo y coordenadas para el rango solicitado
        time_index = pd.to_datetime(f['time_index'][start_idx:end_idx].astype(str))
        coords = pd.DataFrame(f['meta'][:])
        nodos_ids = coords.index.values
        
        for var in variables:
            # 1. Leer los datos crudos (Enteros)
            dataset = f[var]
            datos_crudos = dataset[start_idx:end_idx, :]
            
            # 2. Obtener el factor de escala de los atributos (por defecto 1 si no existe)
            scale_factor = dataset.attrs.get('psm_scale_factor', 1.0)
            
            # 3. Aplicar escala y convertir a float32 para la GPU
            datos_reales = (datos_crudos / scale_factor).astype(np.float32)
            datos_extraidos[var] = datos_reales
            
    # Reestructurar en formato largo (Long format) ideal para agrupamientos y PyTorch
    df_list = []
    for var, matriz in datos_extraidos.items():
        df_var = pd.DataFrame(matriz, index=time_index, columns=nodos_ids)
        # Convertir a formato largo: [Fecha, Nodo, Valor]
        df_var = df_var.melt(ignore_index=False, var_name='nodo_id', value_name=var)
        df_var.set_index('nodo_id', append=True, inplace=True)
        df_list.append(df_var)

    # Unir todas las variables en un solo DataFrame
    df_final = pd.concat(df_list, axis=1).reset_index()
    df_final.rename(columns={'level_0': 'timestamp'}, inplace=True)
    
    return df_final, coords

# Ejecución de prueba: Extraer el primer día completo (288 pasos de tiempo)
variables_objetivo = ['ghi', 'air_temperature', 'wind_speed']
ruta_archivo = 'Data/datos_nsrdb/nsrdb_puerto_rico_1998.h5'

df_dia1, metadatos_nodos = extraer_tensor_espaciotemporal(
    h5_path=ruta_archivo, 
    variables=variables_objetivo, 
    start_idx=0, 
    end_idx=288
)

print(df_dia1.head(50))

                   timestamp  nodo_id  ghi  air_temperature  wind_speed
0  1998-01-01 00:00:00+00:00        0  0.0             27.0         0.4
1  1998-01-01 00:05:00+00:00        0  0.0             27.0         0.4
2  1998-01-01 00:10:00+00:00        0  0.0             27.0         0.4
3  1998-01-01 00:15:00+00:00        0  0.0             27.0         0.4
4  1998-01-01 00:20:00+00:00        0  0.0             27.0         0.4
5  1998-01-01 00:25:00+00:00        0  0.0             27.0         0.4
6  1998-01-01 00:30:00+00:00        0  0.0             27.0         0.4
7  1998-01-01 00:35:00+00:00        0  0.0             27.0         0.4
8  1998-01-01 00:40:00+00:00        0  0.0             27.0         0.4
9  1998-01-01 00:45:00+00:00        0  0.0             27.0         0.4
10 1998-01-01 00:50:00+00:00        0  0.0             27.0         0.5
11 1998-01-01 00:55:00+00:00        0  0.0             27.0         0.5
12 1998-01-01 01:00:00+00:00        0  0.0             27.0     

In [4]:
df_dia1.shape

(714240, 5)

In [13]:
import os
import h5py
import pandas as pd
import numpy as np

def procesar_archivo_anual(h5_path, output_dir):
    # Crear el repositorio de salida si no existe
    os.makedirs(output_dir, exist_ok=True)
    
    # Extraer el año del nombre del archivo para automatizar el nombre de salida
    filename = os.path.basename(h5_path)
    year = filename.split('_')[-1].split('.')[0]
    
    # Identificar variables estructurales que NO son series de tiempo
    excluir = ['meta', 'coordinates', 'time_index']
    
    print(f"--- Iniciando procesamiento para el año {year} ---")
    
    with h5py.File(h5_path, 'r') as f:
        # Obtener dinámicamente todas las variables meteorológicas/radiación
        variables = [k for k in f.keys() if k not in excluir]
        
        # Extraer los ejes base
        time_index = pd.to_datetime(f['time_index'][:].astype(str))
        coords = pd.DataFrame(f['meta'][:])
        nodos_ids = coords.index.values
        
        df_final_list = []
        
        for var in variables:
            print(f"Procesando y remuestreando: {var}...")
            dataset = f[var]
            
            # Obtener factor de escala
            scale_factor = dataset.attrs.get('psm_scale_factor', 1.0)
            
            # 1. Cargar matriz completa en RAM y aplicar escala (105,120 x 2,480)
            datos = (dataset[:] / scale_factor).astype(np.float32)
            
            # 2. Convertir a DataFrame ancho para usar el motor de series temporales
            df_var = pd.DataFrame(datos, index=time_index, columns=nodos_ids)
            df_var.index.name = 'timestamp'
            
            # 3. Remuestreo a 1 hora INMEDIATO (reduce a 8,760 x 2,480)
            df_var_1h = df_var.resample('1h').mean()
            
            # 4. Derretir a formato largo (Long format)
            df_largo = df_var_1h.melt(ignore_index=False, var_name='nodo_id', value_name=var)
            df_largo.set_index('nodo_id', append=True, inplace=True)
            
            df_final_list.append(df_largo)
            
    # 5. Concatenar todas las variables procesadas
    print("\nUniendo variables y estructurando dataset final...")
    df_anual = pd.concat(df_final_list, axis=1).reset_index()
    
    # 6. Guardar en formato Parquet en la carpeta Data/Horarias
    out_file = os.path.join(output_dir, f'nsrdb_pr_{year}_1H.parquet')
    df_anual.to_parquet(out_file, engine='pyarrow', index=False)
    
    print(f"Archivo guardado exitosamente: {out_file}")
    return df_anual

# ==========================================
# Ejecución de la prueba
# ==========================================
ruta_entrada = 'Data/datos_nsrdb/nsrdb_puerto_rico_2017.h5'
directorio_salida = 'Data/Horarias'

df_1998 = procesar_archivo_anual(ruta_entrada, directorio_salida)

print("\nVista previa del dataset procesado:")
print(df_1998.head())
print("\nInformación de memoria y estructura:")
print(df_1998.info())

--- Iniciando procesamiento para el año 2017 ---
Procesando y remuestreando: air_temperature...
Procesando y remuestreando: clearsky_dhi...
Procesando y remuestreando: clearsky_dni...
Procesando y remuestreando: clearsky_ghi...
Procesando y remuestreando: dhi...
Procesando y remuestreando: dni...
Procesando y remuestreando: ghi...
Procesando y remuestreando: solar_zenith_angle...
Procesando y remuestreando: surface_albedo...
Procesando y remuestreando: surface_pressure...
Procesando y remuestreando: total_precipitable_water...
Procesando y remuestreando: wind_speed...

Uniendo variables y estructurando dataset final...
Archivo guardado exitosamente: Data/Horarias/nsrdb_pr_2017_1H.parquet

Vista previa del dataset procesado:
                  timestamp  nodo_id  air_temperature  clearsky_dhi  \
0 2017-01-01 00:00:00+00:00        0             26.0           0.0   
1 2017-01-01 01:00:00+00:00        0             26.0           0.0   
2 2017-01-01 02:00:00+00:00        0             26.0

In [14]:
import h5py

ruta_entrada = 'Data/datos_nsrdb/nsrdb_puerto_rico_2017.h5'

with h5py.File(ruta_entrada, 'r') as f:
    print("Claves raíz reales en el archivo:")
    print(list(f.keys()))
    
    # Comprobar si existen subgrupos ocultos o atributos especiales
    if 'meta' in f:
        print("\nColumnas disponibles en 'meta':")
        print(f['meta'].dtype.names)


Claves raíz reales en el archivo:
['air_temperature', 'clearsky_dhi', 'clearsky_dni', 'clearsky_ghi', 'coordinates', 'dhi', 'dni', 'ghi', 'meta', 'solar_zenith_angle', 'surface_albedo', 'surface_pressure', 'time_index', 'total_precipitable_water', 'wind_speed']

Columnas disponibles en 'meta':
('latitude', 'longitude', 'elevation')


In [8]:
df_1998.columns

Index(['timestamp', 'nodo_id', 'air_temperature', 'clearsky_dhi',
       'clearsky_dni', 'clearsky_ghi', 'dhi', 'dni', 'ghi',
       'solar_zenith_angle', 'surface_albedo', 'surface_pressure',
       'total_precipitable_water', 'wind_speed'],
      dtype='str')

In [9]:
import glob
import h5py
import os

def auditar_esquema_archivos(directorio_entrada='./datos_nsrdb/*.h5'):
    """
    Escanea todos los archivos HDF5 en un directorio y verifica si 
    contienen exactamente los mismos datasets (columnas).
    """
    archivos = sorted(glob.glob(directorio_entrada))
    
    if not archivos:
        print("No se encontraron archivos .h5 en el directorio.")
        return

    print(f"Auditando esquema de {len(archivos)} archivos...\n")
    
    # 1. Tomar el primer archivo como la "Verdad Base" (Baseline)
    archivo_base = archivos[0]
    nombre_base = os.path.basename(archivo_base)
    
    with h5py.File(archivo_base, 'r') as f:
        columnas_base = set(f.keys())
        
    print(f"Archivo de referencia: {nombre_base}")
    print(f"Total de variables base: {len(columnas_base)}")
    print(f"Estructura: {sorted(list(columnas_base))}\n")
    
    archivos_con_diferencias = {}
    
    # 2. Iterar sobre el resto de los archivos y comparar conjuntos
    for ruta in archivos[1:]:
        nombre_archivo = os.path.basename(ruta)
        
        with h5py.File(ruta, 'r') as f:
            columnas_actuales = set(f.keys())
            
        # Comparación de conjuntos (Sets)
        if columnas_actuales != columnas_base:
            faltantes = columnas_base - columnas_actuales
            extras = columnas_actuales - columnas_base
            archivos_con_diferencias[nombre_archivo] = {
                'faltantes': faltantes,
                'extras': extras
            }
            
    # 3. Reporte Final
    print("-" * 60)
    if not archivos_con_diferencias:
        print("✅ EXCELENTE: Todos los archivos tienen exactamente la misma estructura.")
        print("Puedes ejecutar el procesamiento masivo sin riesgo de errores.")
    else:
        print("⚠️ ADVERTENCIA: Se encontraron inconsistencias en la estructura temporal:")
        for arch, diff in archivos_con_diferencias.items():
            print(f"\n[{arch}]")
            if diff['faltantes']:
                print(f"  ❌ Faltan: {diff['faltantes']}")
            if diff['extras']:
                print(f"  ➕ Sobran/Nuevas: {diff['extras']}")

# Ejecutar la auditoría
auditar_esquema_archivos('Data/datos_nsrdb/*.h5')

Auditando esquema de 20 archivos...

Archivo de referencia: nsrdb_puerto_rico_1998.h5
Total de variables base: 15
Estructura: ['air_temperature', 'clearsky_dhi', 'clearsky_dni', 'clearsky_ghi', 'coordinates', 'dhi', 'dni', 'ghi', 'meta', 'solar_zenith_angle', 'surface_albedo', 'surface_pressure', 'time_index', 'total_precipitable_water', 'wind_speed']

------------------------------------------------------------
✅ EXCELENTE: Todos los archivos tienen exactamente la misma estructura.
Puedes ejecutar el procesamiento masivo sin riesgo de errores.


In [10]:
import os
import glob
import time
import h5py
import pandas as pd
import numpy as np

def procesar_archivo_anual(h5_path, output_dir):
    # Crear el repositorio de salida si no existe
    os.makedirs(output_dir, exist_ok=True)
    
    # Extraer el año del nombre del archivo para automatizar el nombre de salida
    filename = os.path.basename(h5_path)
    year = filename.split('_')[-1].split('.')[0]
    
    # Identificar variables estructurales que NO son series de tiempo
    excluir = ['meta', 'coordinates', 'time_index']
    
    print(f"\n[{year}] --- Iniciando procesamiento ---")
    
    with h5py.File(h5_path, 'r') as f:
        # Obtener dinámicamente todas las variables meteorológicas/radiación
        variables = [k for k in f.keys() if k not in excluir]
        
        # Extraer los ejes base
        time_index = pd.to_datetime(f['time_index'][:].astype(str))
        coords = pd.DataFrame(f['meta'][:])
        nodos_ids = coords.index.values
        
        df_final_list = []
        
        for var in variables:
            dataset = f[var]
            
            # Obtener factor de escala
            scale_factor = dataset.attrs.get('psm_scale_factor', 1.0)
            
            # 1. Cargar matriz completa en RAM y aplicar escala
            datos = (dataset[:] / scale_factor).astype(np.float32)
            
            # 2. Convertir a DataFrame ancho
            df_var = pd.DataFrame(datos, index=time_index, columns=nodos_ids)
            df_var.index.name = 'timestamp'
            
            # 3. Remuestreo a 1 hora INMEDIATO
            df_var_1h = df_var.resample('1h').mean()
            
            # 4. Derretir a formato largo
            df_largo = df_var_1h.melt(ignore_index=False, var_name='nodo_id', value_name=var)
            df_largo.set_index('nodo_id', append=True, inplace=True)
            
            df_final_list.append(df_largo)
            
    # 5. Concatenar todas las variables procesadas
    print(f"[{year}] Uniendo variables y estructurando dataset final...")
    df_anual = pd.concat(df_final_list, axis=1).reset_index()
    
    # 6. Guardar en formato Parquet
    out_file = os.path.join(output_dir, f'nsrdb_pr_{year}_1H.parquet')
    df_anual.to_parquet(out_file, engine='pyarrow', index=False)
    
    print(f"[{year}] ✅ Guardado exitosamente: {out_file}")
    
    # Retornamos True solo como indicador de éxito para evitar acumular DataFrames en memoria
    return True

# ==========================================
# Ejecución de Procesamiento Masivo
# ==========================================
if __name__ == "__main__":
    directorio_entrada = 'Data/datos_nsrdb/*.h5'
    directorio_salida = 'Data/Horarias'

    # 1. Encontrar todos los archivos HDF5
    archivos_h5 = sorted(glob.glob(directorio_entrada))
    total_archivos = len(archivos_h5)
    
    if total_archivos == 0:
        print(f"No se encontraron archivos en la ruta: {directorio_entrada}")
    else:
        print(f"Se encontraron {total_archivos} archivos para procesar.")
        print("Iniciando tubería de datos espaciotemporales...\n")
        print("=" * 60)
        
        start_time_total = time.time()

        # 2. Bucle principal de procesamiento
        for idx, ruta_archivo in enumerate(archivos_h5, 1):
            inicio_archivo = time.time()
            
            print(f"Procesando archivo {idx}/{total_archivos}: {os.path.basename(ruta_archivo)}")
            
            # Llamada a la función
            procesar_archivo_anual(ruta_archivo, directorio_salida)
            
            fin_archivo = time.time()
            tiempo_archivo = fin_archivo - inicio_archivo
            print(f"Tiempo iteración: {tiempo_archivo:.2f} segundos")
            print("-" * 60)

        # 3. Resumen final
        tiempo_total_min = (time.time() - start_time_total) / 60
        print("\n" + "=" * 60)
        print(f"🚀 PROCESAMIENTO HISTÓRICO COMPLETADO")
        print(f"Total de años procesados: {total_archivos}")
        print(f"Tiempo total de ejecución: {tiempo_total_min:.2f} minutos")
        print("=" * 60)

Se encontraron 20 archivos para procesar.
Iniciando tubería de datos espaciotemporales...

Procesando archivo 1/20: nsrdb_puerto_rico_1998.h5

[1998] --- Iniciando procesamiento ---
[1998] Uniendo variables y estructurando dataset final...
[1998] ✅ Guardado exitosamente: Data/Horarias/nsrdb_pr_1998_1H.parquet
Tiempo iteración: 32.29 segundos
------------------------------------------------------------
Procesando archivo 2/20: nsrdb_puerto_rico_1999.h5

[1999] --- Iniciando procesamiento ---
[1999] Uniendo variables y estructurando dataset final...
[1999] ✅ Guardado exitosamente: Data/Horarias/nsrdb_pr_1999_1H.parquet
Tiempo iteración: 32.75 segundos
------------------------------------------------------------
Procesando archivo 3/20: nsrdb_puerto_rico_2000.h5

[2000] --- Iniciando procesamiento ---
[2000] Uniendo variables y estructurando dataset final...
[2000] ✅ Guardado exitosamente: Data/Horarias/nsrdb_pr_2000_1H.parquet
Tiempo iteración: 33.76 segundos
----------------------------

### Database de prueba para 2017

In [ ]:
import time
import os
from Utils.puerto_rico_v3_2017.etl_solar import procesar_nsrdb_anual


# Configuración de rutas
# Base física diurna (preprocesado) del pipeline v3 -> es un INTERMEDIO:
# se fusiona luego con la API para producir los finales (filtrado/completo).
archivo_prueba = 'Data/datos_nsrdb/nsrdb_puerto_rico_2017.h5' 
directorio_salida = 'Data/Puerto_Rico_v3_2017/intermedios'
# crear el directorio de salida si no existe
os.makedirs(directorio_salida, exist_ok=True)

# Elección de formato ('parquet' o 'csv')
# formato_elegido = 'parquet' 
formato_elegido = 'csv' 

start_time = time.time()

try:
    # Mandamos a llamar la función
    procesar_nsrdb_anual(
        h5_path=archivo_prueba, 
        output_dir=directorio_salida, 
        formato_salida=formato_elegido
    )
    
    tiempo_total = time.time() - start_time
    print(f"\n🚀 Tarea finalizada con éxito en {tiempo_total:.2f} segundos.")
    
except FileNotFoundError:
    print(f"❌ Error: No se encontró el archivo {archivo_prueba}. Verifica la ruta.")
except Exception as e:
    print(f"❌ Ocurrió un error inesperado durante el procesamiento: {e}") 

In [19]:
# Obtener el peso del archivo generado
nombre_archivo_salida = f"nsrdb_pr_preprocesado_2017.parquet"
ruta_archivo_salida = os.path.join(directorio_salida, nombre_archivo_salida)
if os.path.exists(ruta_archivo_salida):
    peso_bytes = os.path.getsize(ruta_archivo_salida)
    peso_mb = peso_bytes / (1024 * 1024)
    print(f"📁 Archivo generado: {nombre_archivo_salida}")
    print(f"📏 Tamaño del archivo: {peso_mb:.2f} MB")



📁 Archivo generado: nsrdb_pr_preprocesado_2017.parquet
📏 Tamaño del archivo: 117.74 MB


In [ ]:
import pandas as pd
import os

nombre_archivo_salida = f"nsrdb_pr_preprocesado_2017.csv"
directorio_salida = 'Data/Puerto_Rico_v3_2017/intermedios'
ruta_archivo_salida = os.path.join(directorio_salida, nombre_archivo_salida)

# Leer el archivo CSV generado
try:
    df = pd.read_csv(ruta_archivo_salida)
    print(f"✅ Archivo '{nombre_archivo_salida}' cargado exitosamente.")
    print(f"Dimensiones del DataFrame: {df.shape}")
    print("Vista previa de los datos:")
    print(df.head())
except FileNotFoundError:
    print(f"❌ Error: No se encontró el archivo '{nombre_archivo_salida}'. Verifica la ruta.")
except Exception as e:
    print(f"❌ Ocurrió un error inesperado al cargar el archivo: {e}")

# Cuantos nodos únicos hay en el dataset y cuantas filas por nodo
if 'nodo_id' in df.columns:
    nodos_unicos = df['nodo_id'].nunique()
    filas_por_nodo = df.groupby('nodo_id').size().unique()
    print(f"\nNúmero de nodos únicos: {nodos_unicos}")
    print(f"Filas por nodo (debería ser consistente): {filas_por_nodo}")

In [10]:
import pandas as pd
# Leer el dataset procesado para verificación para un nodo específico (ej. nodo_id=0) y un rango de fechas y mostrarlo en un dataframe
dataset_procesado = os.path.join(directorio_salida, f'nsrdb_pr_preprocesado_2017.{formato_elegido}')
if formato_elegido == 'parquet':
    df_verificacion = pd.read_parquet(dataset_procesado, engine='pyarrow')
elif formato_elegido == 'csv':
    df_verificacion = pd.read_csv(dataset_procesado)

# Filtrar por nodo_id=0
df_verificacion = df_verificacion[df_verificacion['nodo_id'] == 0]

print("\nVista previa del dataset procesado:")
df_verificacion.head(48)


Vista previa del dataset procesado:


,timestamp,nodo_id,ghi,dni,dhi,clearsky_ghi,air_temperature,surface_pressure,wind_speed,total_precipitable_water,solar_zenith_angle,hora,mes
0,2017-01-01 11:00:00,0,60.416668,275.500000,25.750000,60.416668,26.000000,1010.000000,7.308332,3.519750,84.550003,11,1
2480,2017-01-01 12:00:00,0,259.333344,663.000000,59.083332,259.333344,26.000000,1010.000000,7.133333,3.531417,72.180000,12,1
4960,2017-01-01 13:00:00,0,357.583344,406.750000,172.916672,466.333344,26.000000,1010.000000,7.008334,3.554833,60.709999,13,1
7440,2017-01-01 14:00:00,0,605.000000,763.916687,122.500000,634.416687,26.000000,1010.000000,6.733334,3.588166,50.869999,14,1
9920,2017-01-01 15:00:00,0,711.416687,809.083313,134.166672,743.750000,26.000000,1010.000000,6.416667,3.627333,43.830002,15,1
12400,2017-01-01 16:00:00,0,703.000000,658.083313,207.750000,784.583313,26.000000,1010.000000,6.041667,3.667333,41.049999,16,1
14880,2017-01-01 17:00:00,0,493.833344,224.500000,328.500000,753.083313,26.000000,1010.000000,5.808333,3.707083,43.369999,17,1
17360,2017-01-01 18:00:00,0,207.250000,7.000000,202.750000,651.583313,26.166666,1010.000000,5.516667,3.739917,50.070000,18,1
19840,2017-01-01 19:00:00,0,397.583344,534.416687,137.000000,489.416656,27.000000,1010.000000,5.133333,3.760416,59.700001,19,1
22320,2017-01-01 20:00:00,0,287.500000,627.416687,76.416664,287.500000,27.000000,1010.000000,4.908333,3.744167,71.059998,20,1


In [12]:
# nombre de las columnas del dataframe
print("\nColumnas del dataset procesado:")
print(df_verificacion.columns)


Columnas del dataset procesado:
Index(['timestamp', 'nodo_id', 'ghi', 'dni', 'dhi', 'clearsky_ghi',
       'air_temperature', 'surface_pressure', 'wind_speed',
       'total_precipitable_water', 'solar_zenith_angle', 'hora', 'mes'],
      dtype='str')


In [11]:
import h5py
import numpy as np
import pandas as pd

def auditar_calculos_hora(h5_path, nodo_id, fecha_hora_str):
    """
    Extrae y muestra los 12 registros originales (intervalos de 5 minutos) 
    para una hora específica y un nodo dado, permitiendo validar matemáticamente 
    los promedios horarios.
    """
    target_time = pd.to_datetime(fecha_hora_str)
    
    print(f"\n{'='*50}")
    print(f"AUDITORÍA DE DATOS ORIGINALES A 5 MINUTOS")
    print(f"Archivo: {h5_path}")
    print(f"Nodo Objetivo: {nodo_id}")
    print(f"Hora Evaluada: {target_time}")
    print(f"{'='*50}\n")
    
    with h5py.File(h5_path, 'r') as f:
        # 1. Encontrar los índices exactos en el arreglo de tiempo
        time_index = pd.to_datetime(f['time_index'][:].astype(str))
        
        # Encontrar donde la fecha y hora coinciden (Minuto 00)
        mask_hora = (time_index.year == target_time.year) & \
                    (time_index.month == target_time.month) & \
                    (time_index.day == target_time.day) & \
                    (time_index.hour == target_time.hour)
        
        indices_hora = np.where(mask_hora)[0]
        
        if len(indices_hora) == 0:
            print("No se encontraron registros para esa hora. Verifica el formato (YYYY-MM-DD HH:MM:SS)")
            return
            
        # Extraer los 12 tiempos exactos
        tiempos_5min = time_index[indices_hora]
        
        # 2. Definir las variables a extraer
        variables_promedio = ['ghi', 'dni', 'dhi', 'clearsky_ghi']
        
        # Crear un diccionario para construir el DataFrame
        data_cruda = {'timestamp_5min': tiempos_5min}
        
        for var in variables_promedio:
            if var in f:
                dataset = f[var]
                scale = dataset.attrs.get('psm_scale_factor', 1.0)
                # Extraemos las 12 filas (tiempo) y 1 columna (nodo)
                valores_crudos = dataset[indices_hora[0]:indices_hora[-1]+1, nodo_id]
                data_cruda[var] = valores_crudos / scale
        
        # Añadir el ángulo cenital para ver el minuto 30
        if 'solar_zenith_angle' in f:
            dataset_z = f['solar_zenith_angle']
            scale_z = dataset_z.attrs.get('psm_scale_factor', 1.0)
            data_cruda['solar_zenith_angle'] = dataset_z[indices_hora[0]:indices_hora[-1]+1, nodo_id] / scale_z
            
    # 3. Mostrar la tabla cruda
    df_crudo = pd.DataFrame(data_cruda)
    print("Tabla Cruda (12 registros de 5 min):")
    print(df_crudo.to_string(index=False))
    
    # 4. Calcular y mostrar los estadísticos simulados del ETL
    print("\n" + "-"*50)
    print("CÁLCULOS QUE DEBIÓ HACER EL ETL_SOLAR:")
    print("-" * 50)
    
    for var in variables_promedio:
        if var in df_crudo.columns:
            promedio_calc = df_crudo[var].mean()
            print(f"-> Promedio de {var.ljust(15)}: {promedio_calc:.6f}")
            
    if 'solar_zenith_angle' in df_crudo.columns:
        # El índice 6 corresponde al minuto 30 (00, 05, 10, 15, 20, 25, *30*)
        min_30_calc = df_crudo['solar_zenith_angle'].iloc[6]
        print(f"-> Valor del minuto 30 para solar_zenith_angle: {min_30_calc:.6f}")
        
    print(f"\n👉 Compara estos resultados finales con la fila de tu DataFrame de 1 hora.")

# ==========================================
# Ejecución
# ==========================================
# Selecciona la primera fila de tu tabla para validar
archivo_original = 'Data/datos_nsrdb/nsrdb_puerto_rico_2017.h5' # Asegúrate de que el año coincida
nodo = 0
hora_a_validar = '2017-01-01 11:00:00'

auditar_calculos_hora(archivo_original, nodo, hora_a_validar)


AUDITORÍA DE DATOS ORIGINALES A 5 MINUTOS
Archivo: Data/datos_nsrdb/nsrdb_puerto_rico_2017.h5
Nodo Objetivo: 0
Hora Evaluada: 2017-01-01 11:00:00

Tabla Cruda (12 registros de 5 min):
           timestamp_5min   ghi   dni  dhi  clearsky_ghi  solar_zenith_angle
2017-01-01 11:00:00+00:00   0.0   0.0  0.0           0.0               89.00
2017-01-01 11:05:00+00:00   0.0   0.0  0.0           0.0               89.00
2017-01-01 11:10:00+00:00  15.0  93.0 13.0          15.0               88.61
2017-01-01 11:15:00+00:00  24.0 154.0 17.0          24.0               87.64
2017-01-01 11:20:00+00:00  35.0 215.0 22.0          35.0               86.62
2017-01-01 11:25:00+00:00  47.0 272.0 26.0          47.0               85.59
2017-01-01 11:30:00+00:00  61.0 324.0 30.0          61.0               84.55
2017-01-01 11:35:00+00:00  76.0 371.0 34.0          76.0               83.51
2017-01-01 11:40:00+00:00  92.0 414.0 37.0          92.0               82.46
2017-01-01 11:45:00+00:00 108.0 453.0 41.0   

In [ ]:
import pandas as pd

# Ruta al archivo del primer nodo generado por la descarga masiva
ruta_archivo = 'Data/Puerto_Rico_v4_2017/crudos_api/2017_V4_Completo/nodo_0_2017_v4.parquet'

print(f"Lectura de archivo: {ruta_archivo}\n")

try:
    # Cargar el archivo usando el motor optimizado
    df = pd.read_parquet(ruta_archivo, engine='pyarrow')
    
    print("=" * 80)
    print("VISTA PREVIA: PRIMERAS 24 HORAS (1 de Enero de 2017)")
    print("=" * 80)
    
    # Mostrar las primeras 24 filas completas sin que Pandas oculte columnas
    print(df.head(24).to_string(index=False))
    
    print("\n" + "=" * 80)
    print(f"Estructura validada: {len(df):,} filas x {len(df.columns)} columnas")
    print("=" * 80)

except FileNotFoundError:
    print(f"❌ Error: El archivo no existe en {ruta_archivo}.")
    print("Verifica si la descarga masiva ya terminó de procesar el nodo 0.")
except Exception as e:
    print(f"❌ Ocurrió un error inesperado al leer el archivo: {e}")

In [ ]:
import pandas as pd

# Ruta al archivo del primer nodo generado por la descarga masiva
ruta_archivo = 'Data/Puerto_Rico_v3_2017/crudos_api/2017/nodo_10_2017.parquet'

print(f"Lectura de archivo: {ruta_archivo}\n")

try:
    # Cargar el archivo usando el motor optimizado
    df = pd.read_parquet(ruta_archivo, engine='pyarrow')
    
    print("=" * 80)
    print("VISTA PREVIA: PRIMERAS 24 HORAS (1 de Enero de 2017)")
    print("=" * 80)
    
    # Mostrar las primeras 24 filas completas sin que Pandas oculte columnas
    print(df.head(24).to_string(index=False))
    
    print("\n" + "=" * 80)
    print(f"Estructura validada: {len(df):,} filas x {len(df.columns)} columnas")
    print("=" * 80)

except FileNotFoundError:
    print(f"❌ Error: El archivo no existe en {ruta_archivo}.")
    print("Verifica si la descarga masiva ya terminó de procesar el nodo 0.")
except Exception as e:
    print(f"❌ Ocurrió un error inesperado al leer el archivo: {e}")

In [ ]:
from Utils.mapa_interactivo import mapa_nodos

# Mapa con zoom + imagen satelital (Esri) para revisar si están sobre agua.
# Rojo = nodos a revisar (clic para ver nodo_id / msnm) · Azul = vecinos (contexto).
# Estos son los 4 nodos con msnm==0 que quedaron tras el filtro de 117:
mapa_nodos([2603, 2604, 2650, 2701])